# LAB 10 — Tokenization question + context và căn chỉnh span

1. Định dạng input `[CLS] question [SEP] context [SEP]`: `input_ids`, `attention_mask`, `token_type_ids` của ba tokenizer khác nhau;
2. Calculation Exercise 1 và 3, kiểm tra bằng tokenizer thật;
3. character index, word index, token index;
4. `char_to_token_span` (trong `qa_model.py`) với các trường hợp: đáp án bắt đầu / kết thúc giữa token, bị cắt, nằm ngoài window;
5. sliding window: `offset_mapping`, `overflow_to_sample_mapping`, `stride`; tokenizer so với tự cắt window, kiểm tra trên 30 000 câu hỏi train.

In [1]:
import os, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"; os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from transformers import AutoTokenizer
import qa_model as qm
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 100)
tok = AutoTokenizer.from_pretrained(qm.MODEL_NAME)
Q, C = "What is the capital of France?", "The capital of France is Paris."

## 1. Input của QA: ba tokenizer

In [2]:
from transformers import AutoModelForQuestionAnswering
import inspect
for name in ["distilbert-base-uncased", "bert-base-uncased", "roberta-base"]:
    t = AutoTokenizer.from_pretrained(name)
    enc = t(Q, C)
    arch = AutoModelForQuestionAnswering.from_config(__import__("transformers").AutoConfig.from_pretrained(name))
    uses_tt = "token_type_ids" in inspect.signature(arch.forward).parameters
    print(f"--- {name}: tokenizer trả về {list(enc.keys())} | model.forward nhận token_type_ids? {uses_tt}")
    print("   tokens        :", t.convert_ids_to_tokens(enc["input_ids"]))
    print("   input_ids     :", enc["input_ids"])
    print("   attention_mask:", enc["attention_mask"])
    if "token_type_ids" in enc: print("   token_type_ids:", enc["token_type_ids"])
    print("   sequence_ids  :", enc.sequence_ids())
    del arch

--- distilbert-base-uncased: tokenizer trả về ['input_ids', 'token_type_ids', 'attention_mask'] | model.forward nhận token_type_ids? False
   tokens        : ['[CLS]', 'what', 'is', 'the', 'capital', 'of', 'france', '?', '[SEP]', 'the', 'capital', 'of', 'france', 'is', 'paris', '.', '[SEP]']
   input_ids     : [101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 1996, 3007, 1997, 2605, 2003, 3000, 1012, 102]
   attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
   token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
   sequence_ids  : [None, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, None]


--- bert-base-uncased: tokenizer trả về ['input_ids', 'token_type_ids', 'attention_mask'] | model.forward nhận token_type_ids? True
   tokens        : ['[CLS]', 'what', 'is', 'the', 'capital', 'of', 'france', '?', '[SEP]', 'the', 'capital', 'of', 'france', 'is', 'paris', '.', '[SEP]']
   input_ids     : [101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 1996, 3007, 1997, 2605, 2003, 3000, 1012, 102]
   attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
   token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
   sequence_ids  : [None, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, None]


--- roberta-base: tokenizer trả về ['input_ids', 'attention_mask'] | model.forward nhận token_type_ids? True
   tokens        : ['<s>', 'What', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof', 'ĠFrance', '?', '</s>', '</s>', 'The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis', 'ĠParis', '.', '</s>']
   input_ids     : [0, 2264, 16, 5, 812, 9, 1470, 116, 2, 2, 133, 812, 9, 1470, 16, 2201, 4, 2]
   attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
   sequence_ids  : [None, 0, 0, 0, 0, 0, 0, 0, None, None, 1, 1, 1, 1, 1, 1, 1, None]


- **BERT** dùng `token_type_ids` (segment embedding): 0 cho `[CLS] question [SEP]`, 1 cho `context [SEP]`. Model cộng một embedding riêng cho mỗi segment.
- **DistilBERT** (model dùng trong lab): tokenizer trong bản `transformers` này vẫn trả về `token_type_ids`, nhưng `DistilBertForQuestionAnswering.forward` **không nhận** tham số này, vì DistilBERT đã bỏ segment embedding. Model phân biệt question và context chỉ nhờ vị trí của `[SEP]`. Vì vậy `qa_model.py` chỉ đưa `input_ids` và `attention_mask` vào model.
- **RoBERTa** không có `token_type_ids`; dùng `<s> question </s></s> context </s>`.

Để biết token nào thuộc context, code dùng `sequence_ids()` (0 = question, 1 = context, None = special) thay vì `token_type_ids`, nên chạy được với cả ba loại model.

## 2. Calculation Exercise 1 và 3

In [3]:
enc = tok(Q, C, return_offsets_mapping=True)
seq = enc.sequence_ids()
ctx_tok = [i for i, s in enumerate(seq) if s == 1]
df = pd.DataFrame({"token": tok.convert_ids_to_tokens(enc["input_ids"]), "token index": range(len(seq)), "sequence_id": seq,
                   "offset (ký tự)": enc["offset_mapping"],
                   "context token index": [ctx_tok.index(i) if i in ctx_tok else "" for i in range(len(seq))]})
df.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
token,[CLS],what,is,the,capital,of,france,?,[SEP],the,capital,of,france,is,paris,.,[SEP]
token index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
sequence_id,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN
offset (ký tự),"(0, 0)","(0, 4)","(5, 7)","(8, 11)","(12, 19)","(20, 22)","(23, 29)","(29, 30)","(0, 0)","(0, 3)","(4, 11)","(12, 14)","(15, 21)","(22, 24)","(25, 30)","(30, 31)","(0, 0)"
context token index,,,,,,,,,,0,1,2,3,4,5,6,


In [4]:
for ans in ["Paris", "capital of France"]:
    a_s = C.index(ans); a_e = a_s + len(ans)
    s, e = qm.char_to_token_span(enc["offset_mapping"], seq, a_s, a_e)
    print(f"answer {ans!r}: ký tự đầu {a_s}, ký tự cuối {a_e - 1}, span ký tự [{a_s}, {a_e}) = {C[a_s:a_e]!r}")
    print(f"   → token {tok.convert_ids_to_tokens(enc['input_ids'][s:e + 1])}: start = {s}, end = {e} trong input đầy đủ; "
          f"= ({ctx_tok.index(s)}, {ctx_tok.index(e)}) nếu chỉ đếm token của context")
print("C[25:30] =", repr(C[25:30]), "| C[25] =", repr(C[25]), "| C[29] =", repr(C[29]))

answer 'Paris': ký tự đầu 25, ký tự cuối 29, span ký tự [25, 30) = 'Paris'
   → token ['paris']: start = 14, end = 14 trong input đầy đủ; = (5, 5) nếu chỉ đếm token của context
answer 'capital of France': ký tự đầu 4, ký tự cuối 20, span ký tự [4, 21) = 'capital of France'
   → token ['capital', 'of', 'france']: start = 10, end = 12 trong input đầy đủ; = (1, 3) nếu chỉ đếm token của context
C[25:30] = 'Paris' | C[25] = 'P' | C[29] = 's'


- **Exercise 1:** nếu chỉ đếm context `[the, capital, of, france, is, paris, .]` thì "Paris" có start = end = 5, còn "capital of France" có start = 1, end = 3. Trong input thật, context đứng sau `[CLS]` + 7 token câu hỏi + `[SEP]` (9 token), nên cộng thêm 9: "Paris" là (14, 14), "capital of France" là (10, 12). Model dự đoán theo chỉ số trong input đầy đủ.
- **Exercise 3:** "Paris" chiếm ký tự 25 đến 29, span ký tự [25, 30) (đầu mở, cuối đóng), ứng với 1 token `paris` (offset (25, 30)), start token = end token = 14. Số 25 là vị trí ký tự, 14 là vị trí token: hai hệ đếm khác nhau.

## 3. character index, word index, token index

In [5]:
C2 = "The company was founded by Steve Wozniak and Steve Jobs in 1976."
enc2 = tok("Who founded the company?", C2, return_offsets_mapping=True)
seq2, words = enc2.sequence_ids(), C2.split()
word_start = [C2.index(w, sum(len(x) + 1 for x in words[:i])) for i, w in enumerate(words)]
rows = []
for i, (s, e) in enumerate(enc2["offset_mapping"]):
    if seq2[i] != 1: continue
    wi = max(k for k, ws in enumerate(word_start) if ws <= s)
    rows.append({"token index": i, "token": tok.convert_ids_to_tokens(enc2["input_ids"][i]), "char span": (s, e), "text": C2[s:e], "word index": wi, "word": words[wi]})
pd.DataFrame(rows).set_index("token index").T

token index,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
token,the,company,was,founded,by,steve,wo,##z,##nia,##k,and,steve,jobs,in,1976,.
char span,"(0, 3)","(4, 11)","(12, 15)","(16, 23)","(24, 26)","(27, 32)","(33, 35)","(35, 36)","(36, 39)","(39, 40)","(41, 44)","(45, 50)","(51, 55)","(56, 58)","(59, 63)","(63, 64)"
text,The,company,was,founded,by,Steve,Wo,z,nia,k,and,Steve,Jobs,in,1976,.
word index,0,1,2,3,4,5,6,6,6,6,7,8,9,10,11,11
word,The,company,was,founded,by,Steve,Wozniak,Wozniak,Wozniak,Wozniak,and,Steve,Jobs,in,1976.,1976.


"Wozniak" là **1 word** (word index 6), bắt đầu ở **ký tự 33**, nhưng là **3 token** (`wo ##z ##nia ##k`, tùy tokenizer) ở token index 13 trở đi. Dataset chỉ cho `answer_start` theo ký tự; model chỉ biết token. Offset mapping là cầu nối duy nhất giữa hai hệ.

## 4. Các trường hợp khó của alignment (mục 16 của đề)

In [6]:
a_s = C2.index("Wozniak")
cases = [("đáp án trọn token", a_s, a_s + len("Wozniak")),
         ("bắt đầu giữa token (\"zniak\")", a_s + 2, a_s + len("Wozniak")),
         ("kết thúc giữa token (\"Wozn\")", a_s, a_s + 4),
         ("nhiều word: \"Steve Wozniak\"", C2.index("Steve"), a_s + len("Wozniak"))]
for name, s, e in cases:
    ts, te = qm.char_to_token_span(enc2["offset_mapping"], seq2, s, e)
    print(f"{name:32} ký tự [{s}, {e}) {C2[s:e]!r:18} → token ({ts}, {te}) = {tok.convert_ids_to_tokens(enc2['input_ids'][ts:te + 1])}"
          f" → cắt lại theo offset: {C2[enc2['offset_mapping'][ts][0]:enc2['offset_mapping'][te][1]]!r}")
print("\n→ Đáp án bắt đầu / kết thúc giữa token được mở rộng ra cả token. Khi dự đoán, text được cắt từ context gốc theo offset, "
      "nên giữ đúng chữ hoa, dấu câu, khoảng trắng (không ghép lại từ các mảnh ##).")

đáp án trọn token                ký tự [33, 40) 'Wozniak'          → token (13, 16) = ['wo', '##z', '##nia', '##k'] → cắt lại theo offset: 'Wozniak'
bắt đầu giữa token ("zniak")     ký tự [35, 40) 'zniak'            → token (14, 16) = ['##z', '##nia', '##k'] → cắt lại theo offset: 'zniak'
kết thúc giữa token ("Wozn")     ký tự [33, 37) 'Wozn'             → token (13, 15) = ['wo', '##z', '##nia'] → cắt lại theo offset: 'Woznia'
nhiều word: "Steve Wozniak"      ký tự [27, 40) 'Steve Wozniak'    → token (12, 16) = ['steve', 'wo', '##z', '##nia', '##k'] → cắt lại theo offset: 'Steve Wozniak'

→ Đáp án bắt đầu / kết thúc giữa token được mở rộng ra cả token. Khi dự đoán, text được cắt từ context gốc theo offset, nên giữ đúng chữ hoa, dấu câu, khoảng trắng (không ghép lại từ các mảnh ##).


In [7]:
enc3 = tok("Who founded the company?", C2, truncation="only_second", max_length=16, stride=4,
           return_overflowing_tokens=True, return_offsets_mapping=True)
print("max_length 16, stride 4 → số window:", len(enc3["input_ids"]), "| overflow_to_sample_mapping:", enc3["overflow_to_sample_mapping"])
for name, s, e in [("Steve Wozniak", C2.index("Steve"), a_s + 7), ("1976", C2.index("1976"), C2.index("1976") + 4)]:
    print(f"\nđáp án {name!r}:")
    for k in range(len(enc3["input_ids"])):
        sq = enc3.sequence_ids(k); off = enc3["offset_mapping"][k]
        ctx = [i for i, x in enumerate(sq) if x == 1]
        ts, te = qm.char_to_token_span(off, sq, s, e)
        print(f"  window {k}: context ký tự [{off[ctx[0]][0]}, {off[ctx[-1]][1]}) {C2[off[ctx[0]][0]:off[ctx[-1]][1]]!r:40} → "
              + (f"start/end = ({ts}, {te})" if ts else "(0, 0): không chứa trọn đáp án → nhãn [CLS]"))

max_length 16, stride 4 → số window: 3 | overflow_to_sample_mapping: [0, 0, 0]

đáp án 'Steve Wozniak':
  window 0: context ký tự [0, 36) 'The company was founded by Steve Woz'   → (0, 0): không chứa trọn đáp án → nhãn [CLS]
  window 1: context ký tự [24, 50) 'by Steve Wozniak and Steve'             → start/end = (8, 12)
  window 2: context ký tự [36, 64) 'niak and Steve Jobs in 1976.'           → (0, 0): không chứa trọn đáp án → nhãn [CLS]

đáp án '1976':
  window 0: context ký tự [0, 36) 'The company was founded by Steve Woz'   → (0, 0): không chứa trọn đáp án → nhãn [CLS]
  window 1: context ký tự [24, 50) 'by Steve Wozniak and Steve'             → (0, 0): không chứa trọn đáp án → nhãn [CLS]
  window 2: context ký tự [36, 64) 'niak and Steve Jobs in 1976.'           → start/end = (13, 13)


- Window 0 và 1 chồng lên nhau 4 token (`stride`), nên một đáp án bị cắt ở cuối window này vẫn nằm trọn trong window sau.
- Window không chứa trọn đáp án (bị cắt dở hoặc nằm hẳn bên ngoài) nhận nhãn start = end = 0, tức vị trí `[CLS]`. Khi dự đoán, `postprocess` không bao giờ chọn span ngoài context, và lấy span có điểm cao nhất trên **tất cả** window của câu hỏi.

## 5. Sliding window: tokenizer so với tự cắt, kiểm tra trên toàn bộ 30 000 câu hỏi train

Cách đề gợi ý (`return_overflowing_tokens=True, stride=...`) được so với `qm.encode_windows`: tokenize riêng question và context, rồi tự cắt context thành các đoạn L token, đoạn sau bắt đầu sau đoạn trước L − stride token, cho tới khi phủ hết context.

In [8]:
data = qm.load_squad()
exs = data["train"]
def hf_windows(T, stride):
    enc = tok([e["question"].strip() for e in exs], [e["context"] for e in exs], truncation="only_second", max_length=T, stride=stride,
              return_overflowing_tokens=True, return_offsets_mapping=True)
    return {"input_ids": enc["input_ids"], "offset_mapping": enc["offset_mapping"], "overflow_to_sample_mapping": enc["overflow_to_sample_mapping"],
            "sequence_ids": [enc.sequence_ids(k) for k in range(len(enc["input_ids"]))]}
rows = []
for T, stride in [(128, 64), (256, 128), (384, 128)]:
    for method, enc in [("tokenizer", hf_windows(T, stride)), ("encode_windows", qm.encode_windows(tok, exs, T, stride))]:
        m = enc["overflow_to_sample_mapping"]
        has = np.zeros(len(exs), bool); covered = np.zeros(len(exs)); ok = found = mid = 0
        for k in range(len(m)):
            e = exs[m[k]]; a_s = e["answers"]["answer_start"][0]; a_e = a_s + len(e["answers"]["text"][0])
            off, sq = enc["offset_mapping"][k], enc["sequence_ids"][k]
            ctx = [i for i, x in enumerate(sq) if x == 1]
            covered[m[k]] = max(covered[m[k]], off[ctx[-1]][1])
            s, t = qm.char_to_token_span(off, sq, a_s, a_e)
            if s:
                has[m[k]] = True; found += 1
                ok += qm.normalize_answer(e["context"][off[s][0]:off[t][1]]) == qm.normalize_answer(e["answers"]["text"][0])
                mid += off[s][0] != a_s or off[t][1] != a_e
        full = np.array([covered[i] >= len(e["context"].rstrip()) for i, e in enumerate(exs)])
        rows.append({"max_length": T, "stride": stride, "cách cắt": method, "features": len(m), "window / câu hỏi": len(m) / len(exs),
                     "% câu hỏi được phủ hết context": full.mean() * 100, "% câu hỏi có window chứa trọn đáp án": has.mean() * 100,
                     "% span cắt lại khớp gold": ok / found * 100, "% đáp án không trùng biên token": mid / found * 100})
pd.DataFrame(rows).set_index(["max_length", "cách cắt"]).round(2)

stride  features  window / câu hỏi  % câu hỏi được phủ hết context  % câu hỏi có window chứa trọn đáp án  % span cắt lại khớp gold  % đáp án không trùng biên token
max_length cách cắt                                                                                                                                                                           
128        tokenizer           64     52824              1.76                           37.27                                 86.46                     99.72                             0.28
           encode_windows      64     72988              2.43                          100.00                                100.00                     99.72                             0.28
256        tokenizer          128     32902              1.10                           92.74                                 99.25                     99.71                             0.29
           encode_windows     128     33362              1.11                          100.00                                100.00                     99.71                             0.29
384        tokenizer          128     30307              1.01                           99.23                                 99.94                     99.71                             0.29
           encode_windows     128     30319              1.01                          100.00                                100.00                     99.71                             0.29

In [9]:
e = next(x for x in exs if len(x["context"]) > 1200)
enc = tok(e["question"].strip(), e["context"], truncation="only_second", max_length=128, stride=64, return_overflowing_tokens=True, return_offsets_mapping=True)
print(f"Ví dụ: context {len(e['context'])} ký tự, max_length 128, stride 64")
for k in range(len(enc["input_ids"])):
    sq = enc.sequence_ids(k); c = [i for i, x in enumerate(sq) if x == 1]
    print(f"  tokenizer window {k}: {len(enc['input_ids'][k])} token, context ký tự [{enc['offset_mapping'][k][c[0]][0]}, {enc['offset_mapping'][k][c[-1]][1]})")
mine = qm.encode_windows(tok, [e], 128, 64)
for k in range(len(mine["input_ids"])):
    sq = mine["sequence_ids"][k]; c = [i for i, x in enumerate(sq) if x == 1]
    print(f"  encode_windows {k}: {len(mine['input_ids'][k])} token, context ký tự [{mine['offset_mapping'][k][c[0]][0]}, {mine['offset_mapping'][k][c[-1]][1]})")

Ví dụ: context 1405 ký tự, max_length 128, stride 64
  tokenizer window 0: 128 token, context ký tự [0, 626)
  tokenizer window 1: 92 token, context ký tự [248, 708)
  encode_windows 0: 128 token, context ký tự [0, 626)
  encode_windows 1: 128 token, context ký tự [248, 904)
  encode_windows 2: 128 token, context ký tự [532, 1174)
  encode_windows 3: 120 token, context ký tự [825, 1405)


- **Lỗi của tokenizer:** với `transformers` 5.17, `return_overflowing_tokens=True` cho cặp (question, context) chỉ sinh tối đa 2 window. Window thứ hai dừng giữa chừng, phần context phía sau bị bỏ mất mà không có cảnh báo. Ở max_length = 128, nhiều câu hỏi không có window nào chứa đáp án, nên model không bao giờ thấy đáp án đúng. Vì vậy toàn bộ lab dùng `qm.encode_windows`, phủ hết context và đảm bảo 100% câu hỏi có window chứa trọn đáp án.
- Span cắt lại từ offset khớp gold sau chuẩn hóa ở gần như mọi window. Phần nhỏ không khớp là đáp án bắt đầu / kết thúc giữa token, bị mở rộng ra cả token.
- T càng nhỏ thì càng nhiều window, và phần lớn window không chứa đáp án: model phải học thêm việc trỏ về `[CLS]`.